# 03 - Experiment Tracking with MLflow

This notebook mirrors the model-selection workflow from notebook 02: tune Logistic Regression, Random Forest, and XGBoost with five-fold stratified cross-validation, track every tested hyperparameter combination in MLflow, and save the CV-selected champion for the next steps.

In [1]:
from pathlib import Path
import sys

import mlflow
import mlflow.onnx
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Could not find the project root containing the src folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import download_dataset, load_dataset
from src.feature_engineering import build_preprocessor, prepare_features
from src.onnx_utils import convert_pipeline_to_onnx, save_onnx_model

DATASET_PATH = PROJECT_ROOT / "data" / "heart_disease.csv"
MODEL_PATH = PROJECT_ROOT / "artifacts" / "heart_disease_model.onnx"
mlflow.set_tracking_uri(f"sqlite:///{(PROJECT_ROOT / 'notebooks' / 'mlflow.db').as_posix()}")
mlflow.set_experiment("heart_disease_assignment")

download_dataset(DATASET_PATH)
df = load_dataset(DATASET_PATH)
X, y = prepare_features(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# One stratified holdout split, five-fold CV, and the same search grids.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    "Logistic Regression": Pipeline(
        [
            ("preprocessor", build_preprocessor(X_train)),
            ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
        ]
    ),
    "Random Forest": Pipeline(
        [
            ("preprocessor", build_preprocessor(X_train)),
            ("classifier", RandomForestClassifier(random_state=42)),
        ]
    ),
    "XGBoost": Pipeline(
        [
            ("preprocessor", build_preprocessor(X_train)),
            (
                "classifier",
                XGBClassifier(eval_metric="logloss", random_state=42, n_jobs=1),
            ),
        ]
    ),
}
parameter_grids = {
    "Logistic Regression": {"classifier__C": [0.01, 0.1, 1, 10]},
    "Random Forest": {
        "classifier__n_estimators": [100, 200],
        "classifier__max_depth": [None, 8],
        "classifier__min_samples_leaf": [1, 2],
    },
    "XGBoost": {
        "classifier__n_estimators": [100, 200],
        "classifier__max_depth": [2, 3],
        "classifier__learning_rate": [0.03, 0.1],
    },
}
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
}

searches = {}
comparison_rows = []
candidate_run_ids = {}
with mlflow.start_run(run_name="three_model_five_fold_cv"):
    mlflow.log_params(
        {"cv_folds": 5, "selection_metric": "mean_cv_roc_auc", "random_state": 42}
    )

    for model_name, model in models.items():
        search = GridSearchCV(
            model,
            parameter_grids[model_name],
            scoring=scoring,
            refit="roc_auc",
            cv=cv,
            n_jobs=1,
        )
        search.fit(X_train, y_train)
        searches[model_name] = search

        # Log every tested hyperparameter combination and its mean CV metrics as a child run.
        for candidate_index, params in enumerate(search.cv_results_["params"]):
            with mlflow.start_run(
                run_name=f"{model_name} candidate {candidate_index + 1}", nested=True
            ) as candidate_run:
                candidate_run_ids[(model_name, candidate_index)] = candidate_run.info.run_id
                mlflow.set_tag("model_family", model_name)
                mlflow.set_tag("run_type", "cross_validation_candidate")
                mlflow.set_tag("champion", "false")
                mlflow.log_params(
                    {key.replace("classifier__", ""): str(value) for key, value in params.items()}
                )
                for metric in scoring:
                    mlflow.log_metric(
                        f"cv_{metric}",
                        float(search.cv_results_[f"mean_test_{metric}"][candidate_index]),
                    )

        best_index = search.best_index_
        comparison_rows.append(
            {
                "model": model_name,
                "cv_accuracy": search.cv_results_["mean_test_accuracy"][best_index],
                "cv_precision": search.cv_results_["mean_test_precision"][best_index],
                "cv_recall": search.cv_results_["mean_test_recall"][best_index],
                "cv_roc_auc": search.best_score_,
                "best_parameters": search.best_params_,
            }
        )

    comparison = pd.DataFrame(comparison_rows).sort_values(
        "cv_roc_auc", ascending=False
    )
    display(comparison.round(3))

    # Select the highest-scoring individual candidate among all 20 CV runs.
    champion_cv_roc_auc, champion_name, champion_candidate_index = max(
        (
            (
                float(search.cv_results_["mean_test_roc_auc"][candidate_index]),
                model_name,
                candidate_index,
            )
            for model_name, search in searches.items()
            for candidate_index in range(len(search.cv_results_["params"]))
        ),
        key=lambda candidate: candidate[0],
    )
    champion_search = searches[champion_name]
    if champion_candidate_index != champion_search.best_index_:
        raise RuntimeError("The selected champion candidate does not match its refitted model.")
    champion_run_id = candidate_run_ids[(champion_name, champion_candidate_index)]
    champion_model = champion_search.best_estimator_
    onnx_model = convert_pipeline_to_onnx(champion_model, list(X_train.columns))
    onnx.checker.check_model(onnx_model)

    test_predictions = champion_model.predict(X_test)
    test_probabilities = champion_model.predict_proba(X_test)[:, 1]
    onnx_session = ort.InferenceSession(
        onnx_model.SerializeToString(), providers=["CPUExecutionProvider"]
    )
    onnx_inputs = {
        feature: X_test[[feature]].to_numpy(dtype=np.float32)
        for feature in X_test.columns
    }
    onnx_predictions = onnx_session.run(None, onnx_inputs)[0]
    if not np.array_equal(test_predictions, onnx_predictions):
        raise RuntimeError("ONNX predictions do not match the selected champion pipeline.")

    test_metrics = {
        "test_accuracy": accuracy_score(y_test, test_predictions),
        "test_precision": precision_score(y_test, test_predictions, zero_division=0),
        "test_recall": recall_score(y_test, test_predictions, zero_division=0),
        "test_roc_auc": roc_auc_score(y_test, test_probabilities),
    }

    with mlflow.start_run(run_id=champion_run_id, nested=True):
        mlflow.set_tag("champion", "true")
        mlflow.set_tag("run_type", "champion")
        mlflow.set_tag(
            "mlflow.runName",
            f"CHAMPION - {champion_name} candidate {champion_candidate_index + 1}",
        )
        mlflow.set_tag("champion_selection_metric", "cv_roc_auc")
        mlflow.log_param("champion_model", champion_name)
        mlflow.log_params(
            {
                f"champion_{key.replace('classifier__', '')}": str(value)
                for key, value in champion_search.best_params_.items()
            }
        )
        mlflow.log_metric("champion_cv_roc_auc", champion_cv_roc_auc)
        mlflow.log_metrics({key: float(value) for key, value in test_metrics.items()})
        mlflow.log_dict(
            {"confusion_matrix": confusion_matrix(y_test, test_predictions).tolist()},
            "test_confusion_matrix.json",
        )
        mlflow.onnx.log_model(
            onnx_model,
            name="champion_model",
            onnx_execution_providers=["CPUExecutionProvider"],
        )

    # Save the same verified ONNX model that is logged to MLflow for deployment.
    save_onnx_model(onnx_model, MODEL_PATH)

print(f"Champion MLflow run: {champion_run_id}")
print(f"Champion model: {champion_name} candidate {champion_candidate_index + 1}")
print("Champion parameters:", champion_search.best_params_)
print(f"Mean CV ROC-AUC: {champion_cv_roc_auc:.3f}")
print("Held-out test metrics:")
for metric_name, metric_value in test_metrics.items():
    print(f"{metric_name}: {metric_value:.3f}")
print("Test confusion matrix:")
print(confusion_matrix(y_test, test_predictions))
print(f"Saved ONNX champion model to: {MODEL_PATH}")

2026/10/15 12:41:38 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/15 12:41:38 INFO mlflow.store.db.utils: Updating database tables
2026/10/15 12:41:39 INFO mlflow.tracking.fluent: Experiment with name 'heart_disease_assignment' does not exist. Creating a new experiment.


,model,cv_accuracy,cv_precision,cv_recall,cv_roc_auc,best_parameters
0,Logistic Regression,0.851,0.878,0.792,0.911,{'classifier__C': 0.1}
1,Random Forest,0.802,0.800,0.765,0.897,"{'classifier__max_depth': 8, 'classifier__min_..."
2,XGBoost,0.822,0.828,0.783,0.885,"{'classifier__learning_rate': 0.03, 'classifie..."


Champion MLflow run: aa280c214d374e499b9127d8b34c377d
Champion model: Logistic Regression candidate 2
Champion parameters: {'classifier__C': 0.1}
Mean CV ROC-AUC: 0.911
Held-out test metrics:
test_accuracy: 0.885
test_precision: 0.839
test_recall: 0.929
test_roc_auc: 0.961
Test confusion matrix:
[[28  5]
 [ 2 26]]
Saved ONNX champion model to: d:\M.Tech BITS Pilani WILP\Sem 3\MLOps\MLOps Labs\MLOps-Assignment\artifacts\heart_disease_model.onnx


## Reading the experiment results

The comparison table summarizes each model family's best CV result. The champion is the single candidate run with the highest mean CV ROC-AUC across all 20 hyperparameter combinations; the held-out test data is used only once for its final evaluation. In MLflow, the parent run groups the search and its 20 child runs each record one tested parameter combination and its CV metrics. The winning child is tagged and renamed as the champion and contains its test metrics and ONNX model artifact. The ONNX file includes preprocessing and the selected classifier and is saved locally for deployment. Predictions from the exported ONNX model are checked against the selected scikit-learn pipeline before the artifact is logged and saved.

The final cell starts the local MLflow UI and opens the experiment dashboard in the browser. The UI shows the search parent and its 20 candidate runs, with the champion identified among those candidates by `cv_roc_auc`.

In [2]:
# Start the local MLflow UI once, then open the experiment dashboard in your browser.
import subprocess
import time
import urllib.error
import urllib.request
import webbrowser
from IPython.display import HTML, display

MLFLOW_UI_URL = "http://127.0.0.1:5000"

ui_process = globals().get("_mlflow_ui_process")
if ui_process is None or ui_process.poll() is not None:
    ui_process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "mlflow",
            "ui",
            "--backend-store-uri",
            mlflow.get_tracking_uri(),
            "--host",
            "127.0.0.1",
            "--port",
            "5000",
        ],
        cwd=PROJECT_ROOT,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    globals()["_mlflow_ui_process"] = ui_process

for attempt in range(30):
    if ui_process.poll() is not None:
        raise RuntimeError("The MLflow UI process stopped before it became ready.")
    try:
        with urllib.request.urlopen(MLFLOW_UI_URL, timeout=1) as response:
            if response.status == 200:
                break
    except (urllib.error.URLError, TimeoutError):
        time.sleep(1)
else:
    raise TimeoutError(f"The MLflow UI did not start at {MLFLOW_UI_URL} within 30 seconds.")

webbrowser.open(MLFLOW_UI_URL, new=2)
display(HTML(f'<a href="{MLFLOW_UI_URL}" target="_blank">Open the MLflow experiment dashboard</a>'))
print(f"MLflow UI is running at {MLFLOW_UI_URL}")

MLflow UI is running at http://127.0.0.1:5000
